# Probability Distribution Dashboard

How do you measure uncertainty in geospatial problems? This notebook walks through three
approaches using interactive sliders. Drag a slider, watch the math update, and build
intuition for how confidence changes with more data.

Part of my [35-project AI Engineering Roadmap](https://ai-engineer-roadmap-ee841.web.app/projects).

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import ipywidgets as widgets
from IPython.display import display, clear_output

from distributions import (
    beta_prior, beta_posterior, beta_summary,
    normal_pdf, normal_confidence_interval, simulate_sensor_measurements,
    dirichlet_sample, dirichlet_posterior, dirichlet_summary,
)

plt.style.use('dark_background')
ACCENT = '#34d399'
SECONDARY = '#60a5fa'
TERTIARY = '#f472b6'

---

## 1. How sure are you about a classification?

Imagine you are looking at satellite imagery and trying to figure out what fraction of a
region is forest. You have not looked at the data yet, so you start with a guess. Maybe you
think it is around 50/50. That guess is your **prior belief**.

Then you sample some pixels. Say 7 out of 10 turn out to be forest. Your belief should shift
toward 70%. The Beta distribution is the math that does this shift for you. It takes your
prior guess, combines it with the evidence, and gives you an updated belief (the **posterior**).

The two shape parameters control your starting belief. Equal values = "I have no strong
opinion." Lopsided values = "I already think it is mostly forest (or mostly not)."

**Drag the sliders to set your starting belief and add observations. Watch the curve narrow as you add more data.**

In [ ]:
def plot_beta_update(prior_alpha, prior_beta, observed_forest, observed_total):
    x = np.linspace(0.001, 0.999, 500)
    
    prior_y = beta_prior(prior_alpha, prior_beta, x)
    post_alpha, post_beta = beta_posterior(prior_alpha, prior_beta, observed_forest, observed_total)
    post_y = beta_prior(post_alpha, post_beta, x)
    
    prior_stats = beta_summary(prior_alpha, prior_beta)
    post_stats = beta_summary(post_alpha, post_beta)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Prior
    ax1.fill_between(x, prior_y, alpha=0.3, color=SECONDARY)
    ax1.plot(x, prior_y, color=SECONDARY, linewidth=2)
    ax1.axvline(prior_stats['mean'], color=SECONDARY, linestyle='--', alpha=0.7, label=f'Best guess: {prior_stats["mean"]:.1%}')
    ax1.set_title('Your Starting Belief (Before Data)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Chance a pixel is forest', fontsize=12)
    ax1.set_ylabel('How likely', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.set_xlim(0, 1)
    
    # Posterior
    ax2.fill_between(x, post_y, alpha=0.3, color=ACCENT)
    ax2.plot(x, post_y, color=ACCENT, linewidth=2)
    ax2.axvline(post_stats['mean'], color=ACCENT, linestyle='--', alpha=0.7, label=f'Best guess: {post_stats["mean"]:.1%}')
    ci = post_stats['ci_95']
    ax2.axvspan(ci[0], ci[1], alpha=0.1, color=ACCENT, label=f'95% confident: {ci[0]:.1%} to {ci[1]:.1%}')
    ax2.set_title('Updated Belief (After Data)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Chance a pixel is forest', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.set_xlim(0, 1)
    
    plt.suptitle(f'You sampled {observed_total} pixels and {observed_forest} were forest', 
                 fontsize=12, color='#94a3b8', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print(f'Uncertainty before data: {prior_stats["std"]:.4f}')
    print(f'Uncertainty after data:  {post_stats["std"]:.4f}')
    print(f'Uncertainty reduced by:  {(1 - post_stats["std"]/prior_stats["std"])*100:.1f}%')


beta_out = widgets.Output()
def _run_beta(**kw):
    beta_out.clear_output(wait=True)
    with beta_out:
        plot_beta_update(**kw)

beta_w = dict(
    prior_alpha=widgets.FloatSlider(value=2, min=0.5, max=50, step=0.5, description='Belief strength ( Î±:', style={'description_width': '80px'}),
    prior_beta=widgets.FloatSlider(value=2, min=0.5, max=50, step=0.5, description='Belief strength ( Î²:', style={'description_width': '80px'}),
    observed_forest=widgets.IntSlider(value=7, min=0, max=100, description='Forest pixels found:', style={'description_width': '100px'}),
    observed_total=widgets.IntSlider(value=10, min=1, max=100, description='Total pixels sampled:', style={'description_width': '100px'}),
)
widgets.interactive_output(_run_beta, beta_w)
display(widgets.VBox([*beta_w.values(), beta_out]))

### What to notice

- Start with equal belief strengths (both at 2). The left curve is wide and flat, meaning "could be anything."
- Now sample 7 forest pixels out of 10. The right curve shifts toward 70% and gets narrower. You are more confident now.
- Crank both belief strengths up to 50. Now even 7/10 forest barely moves the curve. A strong starting belief resists new data.
- The shaded green band is your 95% confidence range. It shrinks as you add more data.
- In practice, historical land-cover maps serve as the starting belief. New satellite observations are the evidence.

---

## 2. How many readings do you need?

Geospatial sensors are noisy. A LiDAR sensor measuring elevation might report 252m, then
248m, then 251m for the same spot. Each reading is close to the truth but not exact.

If you average multiple readings, the noise cancels out and you get closer to the real value.
The question is: how much closer? The answer follows a simple rule. Averaging 4 readings
cuts the error in half. Averaging 100 readings cuts it by 10x. The improvement is the square
root of the number of readings, not the number itself.

**Move the sliders to change the true elevation, sensor noise, and number of readings.**

In [ ]:
def plot_normal_uncertainty(true_elevation, noise_std, n_measurements):
    measurements = simulate_sensor_measurements(true_elevation, noise_std, n_measurements)
    sample_mean = measurements.mean()
    ci_low, ci_high = normal_confidence_interval(sample_mean, noise_std, n_measurements)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Individual measurements
    ax1.scatter(range(n_measurements), measurements, alpha=0.5, s=20, color=SECONDARY, label='Individual readings')
    ax1.axhline(true_elevation, color=ACCENT, linewidth=2, label=f'True value: {true_elevation}m')
    ax1.axhline(sample_mean, color=TERTIARY, linewidth=1.5, linestyle='--', label=f'Average: {sample_mean:.1f}m')
    ax1.fill_between(range(n_measurements), ci_low, ci_high, alpha=0.15, color=ACCENT)
    ax1.set_title(f'{n_measurements} Sensor Readings', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Reading number', fontsize=12)
    ax1.set_ylabel('Elevation (meters)', fontsize=12)
    ax1.legend(fontsize=10)
    
    # Distribution of the mean
    standard_error = noise_std / np.sqrt(n_measurements)
    x = np.linspace(true_elevation - 4*noise_std, true_elevation + 4*noise_std, 500)
    
    individual_pdf = normal_pdf(x, true_elevation, noise_std)
    mean_pdf = normal_pdf(x, sample_mean, standard_error)
    
    ax2.plot(x, individual_pdf, color=SECONDARY, alpha=0.5, linewidth=1.5, label=f'One reading (error: +/-Ïƒ={noise_std}m)')
    ax2.fill_between(x, mean_pdf, alpha=0.3, color=ACCENT)
    ax2.plot(x, mean_pdf, color=ACCENT, linewidth=2, label=f'Average of {n_measurements} (error: +/-{standard_error:.1f}m)')
    ax2.axvspan(ci_low, ci_high, alpha=0.1, color=ACCENT)
    ax2.set_title('How Averaging Reduces Uncertainty', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Elevation (meters)', fontsize=12)
    ax2.set_ylabel('How likely', fontsize=12)
    ax2.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f'95% confidence: the true value is between {ci_low:.1f}m and {ci_high:.1f}m')
    print(f'Range width: {ci_high - ci_low:.1f} meters')
    print(f'Average is off by: {abs(sample_mean - true_elevation):.2f}m')


normal_out = widgets.Output()
def _run_normal(**kw):
    normal_out.clear_output(wait=True)
    with normal_out:
        plot_normal_uncertainty(**kw)

normal_w = dict(
    true_elevation=widgets.FloatSlider(value=250, min=0, max=1000, step=10, description='True elevation (m):', style={'description_width': '100px'}),
    noise_std=widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5, description='Sensor noise (m):' Ïƒ (m):', style={'description_width': '100px'}),
    n_measurements=widgets.IntSlider(value=10, min=1, max=200, step=1, description='Number of readings:', style={'description_width': '100px'}),
)
widgets.interactive_output(_run_normal, normal_w)
display(widgets.VBox([*normal_w.values(), normal_out]))

### What to notice

- The **blue curve** shows how spread out a single sensor reading is. This is determined by the hardware and cannot be improved.
- The **green curve** shows how spread out the average is. It gets tighter as you add more readings.
- Try 1 reading, then 4, then 100. Watch the green curve shrink. 4 readings halves the error. 100 readings cuts it by 10x.
- In remote sensing, this is why compositing multiple satellite passes over the same area reduces noise. Each pass is a new reading.

---

## 3. What if there are more than two classes?

The Beta distribution works for two categories (forest vs. not forest). But real land-cover
maps have many classes: forest, water, urban, and so on. The Dirichlet distribution handles
this. It models uncertainty when you have multiple categories whose proportions must add up to 100%.

Each starting belief value represents how strongly you believe that class is present. Low values
mean "I do not know." High values mean "I am fairly confident about this proportion." When you
observe data (count pixels of each type), the values increase and the distribution concentrates
around the observed mix.

**Set your starting beliefs and add pixel observations for each class.**

In [ ]:
CLASSES = ['Forest', 'Water', 'Urban']
COLORS = [ACCENT, SECONDARY, TERTIARY]


def plot_dirichlet(alpha_forest, alpha_water, alpha_urban,
                   obs_forest, obs_water, obs_urban):
    prior_alphas = [alpha_forest, alpha_water, alpha_urban]
    observations = [obs_forest, obs_water, obs_urban]
    post_alphas = dirichlet_posterior(prior_alphas, observations)
    
    prior_samples = dirichlet_sample(prior_alphas, n_samples=5000)
    post_samples = dirichlet_sample(post_alphas, n_samples=5000)
    
    prior_stats = dirichlet_summary(prior_alphas)
    post_stats = dirichlet_summary(post_alphas)
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    
    for i, (cls, color) in enumerate(zip(CLASSES, COLORS)):
        # Prior marginal
        axes[0, i].hist(prior_samples[:, i], bins=50, density=True, alpha=0.6, color=color)
        axes[0, i].axvline(prior_stats['means'][i], color='white', linestyle='--', linewidth=1.5)
        axes[0, i].set_title(f'Before data: {cls}', fontsize=12, fontweight='bold')
        axes[0, i].set_xlabel(f'Chance of {cls.lower()}', fontsize=10)
        axes[0, i].set_xlim(0, 1)
        
        # Posterior marginal
        axes[1, i].hist(post_samples[:, i], bins=50, density=True, alpha=0.6, color=color)
        axes[1, i].axvline(post_stats['means'][i], color='white', linestyle='--', linewidth=1.5)
        axes[1, i].set_title(f'After data: {cls}', fontsize=12, fontweight='bold')
        axes[1, i].set_xlabel(f'Chance of {cls.lower()}', fontsize=10)
        axes[1, i].set_xlim(0, 1)
    
    axes[0, 0].set_ylabel('Before data\n(how likely)', fontsize=11)
    axes[1, 0].set_ylabel('After data\n(how likely)', fontsize=11)
    
    plt.suptitle(
        f'Observed {sum(observations)} pixels: {obs_forest} forest, {obs_water} water, {obs_urban} urban',
        fontsize=12, color='#94a3b8', y=1.02
    )
    plt.tight_layout()
    plt.show()
    
    print('Expected land cover mix (after observations):')
    for cls, mean in zip(CLASSES, post_stats['means']):
        print(f'  {cls}: {mean:.1%}')
    print(f'Overall confidence: {post_stats["concentration"]:.0f} (higher = more certain about the mix)')


dir_out = widgets.Output()
def _run_dir(**kw):
    dir_out.clear_output(wait=True)
    with dir_out:
        plot_dirichlet(**kw)

dir_w = dict(
    alpha_forest=widgets.FloatSlider(value=2, min=0.5, max=20, step=0.5, description='Î± Forest:', style={'description_width': '80px'}),
    alpha_water=widgets.FloatSlider(value=2, min=0.5, max=20, step=0.5, description='Î± Water:', style={'description_width': '80px'}),
    alpha_urban=widgets.FloatSlider(value=2, min=0.5, max=20, step=0.5, description='Î± Urban:', style={'description_width': '80px'}),
    obs_forest=widgets.IntSlider(value=15, min=0, max=50, description='Obs Forest:', style={'description_width': '80px'}),
    obs_water=widgets.IntSlider(value=5, min=0, max=50, description='Obs Water:', style={'description_width': '80px'}),
    obs_urban=widgets.IntSlider(value=3, min=0, max=50, description='Obs Urban:', style={'description_width': '80px'}),
)
widgets.interactive_output(_run_dir, dir_w)
display(widgets.VBox([*dir_w.values(), dir_out]))

### What to notice

- Start with all beliefs at 1 ("I have no idea"). The top row histograms are wide and flat for every class.
- Add observations: 15 forest, 5 water, 3 urban. The bottom row concentrates. Forest peaks around 65%, water around 22%, urban around 13%.
- Increase the starting beliefs to 20. Now even 15 forest observations barely shift the distribution. Strong priors resist data.
- The "overall confidence" number increases as you add data. Higher means the model is more locked in on a specific mix.
- When a machine learning classifier outputs class probabilities like [0.65, 0.22, 0.13], those numbers are not certainties. They are a single sample from a distribution like this.